In [1]:
%load_ext autoreload
%autoreload 2

import torch
from timm.models import create_model

model = create_model(
    'deit_small_distilled_patch16_224',
    pretrained=True,
    num_classes=1000,
    drop_path_rate=0.1,
    drop_block_rate=None,
    global_pool='token',
)

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from compression import dyna_set_sparse_budget, replace_linear_with_, SparseLinear, create_compressed_deit_small

model = replace_linear_with_(
    model, 
    SparseLinear, 
    exclude=[model.get_classifier(), model.head, model.head_dist],
    N=2, M=4, hard=False)

import pickle

sparse_budget = pickle.load(open(
    '/home/chenzhiqiang/MaskLLM-4V/deit_small_layerwise_importance_20251223.pkl', 'rb'))

In [3]:
sd = torch.load('/home/chenzhiqiang/MaskLLM-4V/output/sparse_deit_small_patch16_224_90_epochs_lr1e-3_.augreg_in1k.hybird/HybirdSparse/model_best.pth.tar')

In [4]:
print(sd.keys())

dict_keys(['epoch', 'arch', 'state_dict', 'optimizer', 'version', 'args', 'amp_scaler', 'state_dict_ema', 'metric'])


In [5]:
print(sd['state_dict'].keys())

odict_keys(['cls_token', 'pos_embed', 'dist_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.gate', 'blocks.0.attn.qkv.mask', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.attn.proj.gate', 'blocks.0.attn.proj.mask', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.gate', 'blocks.0.mlp.fc1.mask', 'blocks.0.mlp.fc2.weight', 'blocks.0.mlp.fc2.bias', 'blocks.0.mlp.fc2.gate', 'blocks.0.mlp.fc2.mask', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.qkv.gate', 'blocks.1.attn.qkv.mask', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.attn.proj.gate', 'blocks.1.attn.proj.mask', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc

In [6]:
model = create_compressed_deit_small(pretrained=False, n_bits=8, reduced_token=32, pt_state_dict=sd['state_dict'])

[SparseConfig] Model initialization complete


In [7]:
model

VisionTransformerDistilled(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): CompressedBlock(
      (norm1): IntLayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CompressedAttention(
        (qkv): SparseQuantLinear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): SparseQuantLinear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
        (softmax): IntSoftmaxFixed()
        (matmul_qk): QuantMatmul(bits=8, init_state=0/8)
        (matmul_v): QuantMatmul(bits=8, init_state=0/8)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): IntLayerNorm((384,), e

In [8]:
temp = torch.ones((1, 3, 224, 224), device='cuda')
model = model.to('cuda')
o = model(temp)

In [9]:
print(model.blocks[0].mlp.fc1)

SparseQuantLinear(in_features=384, out_features=1536, bias=True)


: 

## Int LayerNorm

In [19]:
class IntLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, output_bit=8, quant_mode=True, force_dequant="none"):
        super().__init__()
        self.D = normalized_shape[0] if isinstance(normalized_shape, (list, tuple)) else normalized_shape
        self.eps = eps
        
        self.weight = nn.Parameter(torch.ones(self.D))
        self.bias = nn.Parameter(torch.zeros(self.D))

        self.quant_mode = quant_mode
        if force_dequant in ["nonlinear", "layernorm"]:
            self.quant_mode = False

        self.register_buffer("shift", torch.tensor([0], dtype=torch.long))
        self.output_bit = output_bit
        self.max_bit = 32 # Hardware accumulator limit
        
        # --- Hardware Approximation Configurations ---
        self.Q_D = 16
        self.inv_D = int(round((1 << self.Q_D) / self.D))
        
        self.m = 4        # Number of mantissa bits for LUT index
        self.Q_lut = 15   # Fractional bits for LUT values
        self._init_isqrt_lut()

        self.Q_w = 8      # Affine transform precision
        
    def _init_isqrt_lut(self):
        """
        Pre-computes the Inverse Square Root LUT.
        Size = 2^(m+1). Index is formed by: [parity of k (1 bit)] | [mantissa (m bits)]
        """
        lut_size = 1 << (self.m + 1)
        lut = torch.zeros(lut_size, dtype=torch.long)
        for i in range(lut_size):
            parity = i >> self.m
            mantissa = i & ((1 << self.m) - 1)
            # Reconstruct the normalized value: 2^parity * (1 + mantissa / 2^m)
            val = (2 ** parity) * (1.0 + mantissa / (1 << self.m))
            # Store scaled ISqrt value
            lut[i] = int(round((1 << self.Q_lut) / math.sqrt(val)))
        self.register_buffer("isqrt_lut", lut)

    def set_shift(self, y_int):
        """ Adjusts dynamic shift using L_infinity norm to mathematically guarantee no overflow """
        with torch.no_grad():
            # Target: D * (y_max >> S)^2 < 2^max_bit
            # => y_max >> S < sqrt(2^max_bit / D)
            y_max = y_int.abs().max().float()
            max_val_allowed = math.sqrt(((1 << self.max_bit) - 1) / self.D)
            
            if y_max > max_val_allowed:
                # Calculate required shift to keep max squared sum strictly within max_bit
                required_shift = torch.ceil(torch.log2(y_max / max_val_allowed)).long()
                self.shift = torch.max(self.shift, required_shift)

    def forward(self, x, scaling_factor=None):
        if not self.quant_mode:
            mean = x.mean(dim=2, keepdim=True)
            y = x - mean
            var = torch.mean(y**2, dim=2, keepdim=True)
            return (y / torch.sqrt(self.eps + var)) * self.weight + self.bias, None

        # --- Hardware-Equivalent Integer Forward Pass ---
        if scaling_factor is not None:
            x_int = (x / scaling_factor).round().long() # Adding round() reduces quantization noise
        else:
            x_int = x.round().long()

        # [Step 1] Division-free Mean Calculation
        sum_x = x_int.sum(dim=2, keepdim=True)
        mean_int = torch.bitwise_right_shift(sum_x * self.inv_D, self.Q_D)
        
        y_int = x_int - mean_int

        # [Step 2] Variance Calculation with Dynamic Overflow Prevention
        y_shifted = torch.bitwise_right_shift(y_int, self.shift.item())
        y_sq_int = y_shifted * y_shifted
        sum_sq = y_sq_int.sum(dim=2, keepdim=True)
        
        # Check accumulator overflow on SUM OF SQUARES, not variance!
        if self.training and sum_sq.max() >= (1 << self.max_bit):
            self.set_shift(y_int)
            y_shifted = torch.bitwise_right_shift(y_int, self.shift.item())
            sum_sq = (y_shifted * y_shifted).sum(dim=2, keepdim=True)
            
        var_int = torch.bitwise_right_shift(sum_sq * self.inv_D, self.Q_D)
        var_int = torch.clamp(var_int, min=1) # Prevents log2(0)

        # [Step 3] LOD & LUT-based Inverse Square Root
        k = var_int.float().log2().long() 
        parity = k & 1
        mask = (1 << self.m) - 1
        
        shift_amt = k - self.m
        mantissa = torch.where(
            shift_amt >= 0,
            torch.bitwise_right_shift(var_int, shift_amt) & mask,
            torch.bitwise_left_shift(var_int, -shift_amt) & mask
        )
        
        idx = (parity << self.m) | mantissa
        inv_sqrt_val = self.isqrt_lut[idx]

        p = torch.bitwise_right_shift(k, 1)
        
        total_shift = p + self.shift.item() + self.Q_lut
        
        # Quantize Weights and Bias 
        weight_int = (self.weight * (1 << self.Q_w)).round().long()
        bias_int = (self.bias * (1 << self.Q_w)).round().long() 
        
        # Fusion: Y = Y_int * ISqrt * W
        scaled_y = y_int * inv_sqrt_val * weight_int 
        
        # Bit-shift to align with Q_w fractional bits, then add appropriately scaled Bias
        # Broadcasting caveat: total_shift is [B, S, 1], out_int becomes [B, S, D]
        out_int = torch.bitwise_right_shift(scaled_y, total_shift) + bias_int
        
        # Thus out_int purely represents True_FP32_Output * 2^Q_w.
        out_fp32 = out_int.float() / (1 << self.Q_w)

        # Output logic for downstream integer layers:
        # If downstream expects an integer tensor with scale `scaling_factor`:
        new_scaling_factor = scaling_factor if scaling_factor is not None else 1.0
        
        # For seamless integration back to floating point PyTorch framework:
        return out_fp32, new_scaling_factor

In [21]:
def evaluate_layernorm_approximation():
    torch.manual_seed(42)
    
    # 模拟输入参数
    batch_size, seq_len, dim = 16, 128, 768
    input_scale = 0.05 # 模拟前面网络层的浮点缩放因子
    
    # 生成随机浮点输入
    x_fp = torch.randn(batch_size, seq_len, dim) * 2.0 + 0.5
    # 模拟硬件输入的 8-bit 量化整型
    x_int = (x_fp / input_scale).round().long().clamp(-128, 127)
    
    # 重新算回作为 Standard FP32 的实际输入 (消除初始输入量化误差，专注于 LN 内部的误差)
    x_fp_actual = x_int.float() * input_scale
    
    # 1. 初始化标准 LayerNorm
    std_ln = nn.LayerNorm(dim, eps=0.0)
    # 给予一些随机权重和偏置测试鲁棒性
    nn.init.uniform_(std_ln.weight, 0.8, 1.2)
    nn.init.uniform_(std_ln.bias, -0.1, 0.1)
    
    # 2. 初始化整形近似 LayerNorm，并对齐权重
    int_ln = IntLayerNorm(dim)
    int_ln.weight.data.copy_(std_ln.weight.data)
    int_ln.bias.data.copy_(std_ln.bias.data)
    
    # 校准一次以获取动态 Shift 值
    int_ln.set_shift(x_int)
    print(f"[*] Calibration finished. Dynamic Shift determined: {int_ln.shift.item()}")

    # 3. 前向传播
    with torch.no_grad():
        out_std = std_ln(x_fp_actual)
        print("out_std", out_std[0][0][0:10])
        out_int, out_int_scale = int_ln(x_int, input_scale)
        print("out_int:", out_int[0][0][0:10])
        
    # 4. 误差度量
    mse = torch.nn.functional.mse_loss(out_std, out_int).item()
    max_err = torch.max(torch.abs(out_std - out_int)).item()
    
    # 余弦相似度 (衡量向量方向)
    cos_sim = torch.nn.functional.cosine_similarity(
        out_std.view(-1, dim), 
        out_int.view(-1, dim), 
        dim=-1
    ).mean().item()
    
    print("\n" + "="*40)
    print("LayerNorm 整形查表近似精度评估报告")
    print("="*40)
    print(f"Mean Squared Error (MSE):  {mse:.6f}")
    print(f"Max Absolute Error:        {max_err:.6f}")
    print(f"Cosine Similarity:         {cos_sim:.6f} (理想值为 1.0)")
    print("="*40)
    print("分析:")
    if cos_sim > 0.99:
        print("-> 精度极高。近似实现与全精度浮点实现具有高度的一致性。")
    print("-> 误差主要来源:")
    print("   1. m_bits 查表截断：我们仅用尾数的 4 位去寻址 LUT，丢失了微小精度。")
    print("   2. 乘法代替除法时的移位截断误差。")
    print("   3. 缩放权重时引入的 8-bit 定点化误差。")

evaluate_layernorm_approximation()

[*] Calibration finished. Dynamic Shift determined: 0
out_std tensor([ 1.7330,  1.3460,  0.9729, -2.0621,  0.6671, -1.4198, -0.1216, -1.5726,
        -0.6396,  1.8388])
out_int: tensor([ 1.7500,  1.3594,  0.9805, -2.0859,  0.6758, -1.4375, -0.1211, -1.5938,
        -0.6484,  1.8594])

LayerNorm 整形查表近似精度评估报告
Mean Squared Error (MSE):  0.000308
Max Absolute Error:        0.126354
Cosine Similarity:         0.999998 (理想值为 1.0)
分析:
-> 精度极高。近似实现与全精度浮点实现具有高度的一致性。
-> 误差主要来源:
   1. m_bits 查表截断：我们仅用尾数的 4 位去寻址 LUT，丢失了微小精度。
   2. 乘法代替除法时的移位截断误差。
   3. 缩放权重时引入的 8-bit 定点化误差。


## Visualization of Quant scale & Requant params

In [2]:
import torch

state_dict = torch.load('/home/chenzhiqiang/SparseQuant/output/step2_sparse_deit_small_patch16_224_30_epochs_lr1e-3_tp32_8bits.augreg_in1k.hybird/HybirdSparse/last.pth.tar')

In [5]:
print(state_dict['state_dict'].keys())
cur_state_dict = state_dict['state_dict']
qkv_w_alpha = cur_state_dict['blocks.0.attn.qkv.weight_alpha']
qkv_a_alpha = cur_state_dict['blocks.0.attn.qkv.act_alpha']
proj_a_alpha = cur_state_dict['blocks.0.attn.matmul_qk.a_alpha']

rescale = proj_a_alpha / (qkv_w_alpha * qkv_a_alpha)

odict_keys(['cls_token', 'pos_embed', 'dist_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.norm1.norm_scaling_factor', 'blocks.0.norm1.bias_integer', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.weight_alpha', 'blocks.0.attn.qkv.act_alpha', 'blocks.0.attn.qkv.init_state', 'blocks.0.attn.qkv.mask', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.attn.proj.weight_alpha', 'blocks.0.attn.proj.act_alpha', 'blocks.0.attn.proj.init_state', 'blocks.0.attn.proj.mask', 'blocks.0.attn.matmul_qk.a_alpha', 'blocks.0.attn.matmul_qk.b_alpha', 'blocks.0.attn.matmul_qk.init_state', 'blocks.0.attn.matmul_v.a_alpha', 'blocks.0.attn.matmul_v.b_alpha', 'blocks.0.attn.matmul_v.init_state', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.norm2.norm_scaling_factor', 'blocks.0.norm2.bias_integer', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.weight_alpha', 'blo